# 05 · Robustness check: repeated runs with different seeds

Addresses a reviewer question: are the ANN results stable? The binary model is retrained with five seeds and the multiclass model with three. Each run re-splits the data (stratified), recomputes class weights on the training split only, and tunes the threshold on validation.

*Outputs below are from the original Google Colab run used in the paper. To rerun, put the three BCSC files in `data/` (see `data/README.md`).*

## Binary model, five seeds

In [1]:
import os, random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import optuna

def set_global_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 20.9 MB/s eta 0:00:00


In [2]:
df_1 = pd.read_csv('../data/bcsc_risk_factors_summarized1_092020.csv')
df_2 = pd.read_csv('../data/bcsc_risk_factors_summarized2_092020.csv')
df_3 = pd.read_csv('../data/bcsc_risk_factors_summarized3_092020.csv')

df = pd.concat([df_1, df_2, df_3], ignore_index=True)
print("Raw shape:", df.shape)

# Drop unused
df = df.drop(columns=['year', 'count'])
df = df.astype('category')

# Replace 9 -> NaN except age_group_5_years
cols_except_age = df.columns[df.columns != 'age_group_5_years']
df[cols_except_age] = df[cols_except_age].replace(9, np.nan)

# Drop missing target
df = df.dropna(subset=['breast_cancer_history'])

# Fill missing with mode (except target)
for col in df.columns:
    if col != 'breast_cancer_history':
        df[col] = df[col].fillna(df[col].mode()[0])

# One-hot encode race_eth
df = pd.get_dummies(df, columns=['race_eth'], drop_first=True)

# Bool -> int
bool_cols = df.select_dtypes('bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print("After preprocessing shape:", df.shape)
print(df.dtypes.head(10))

Saving bcsc_risk_factors_summarized1_092020.csv to bcsc_risk_factors_summarized1_092020.csv
Saving bcsc_risk_factors_summarized3_092020.csv to bcsc_risk_factors_summarized3_092020.csv
Saving bcsc_risk_factors_summarized2_092020.csv to bcsc_risk_factors_summarized2_092020.csv
Raw shape: (1522340, 13)


After preprocessing shape: (1191438, 15)
age_group_5_years        category
first_degree_hx          category
age_menarche             category
age_first_birth          category
BIRADS_breast_density    category
current_hrt              category
menopaus                 category
bmi_group                category
biophx                   category
breast_cancer_history    category
dtype: object


In [ ]:
# Keep only binary labels
df_bin = df[df['breast_cancer_history'].isin([0, 1])].copy()

y_bin = df_bin['breast_cancer_history'].astype('int32')
X_bin = df_bin.drop(columns=['breast_cancer_history']).astype('float32')

print("Binary X:", X_bin.shape)
print("Binary y counts:\n", y_bin.value_counts())

# ---- Optuna once (use ONE fixed seed split for tuning) ----
TUNE_SEED = 42
set_global_seed(TUNE_SEED)
tf.keras.backend.clear_session()

X_train, X_test, y_train, y_test = train_test_split(
    X_bin, y_bin, test_size=0.2, stratify=y_bin, random_state=TUNE_SEED
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=TUNE_SEED
)

# Class weights
classes = np.unique(y_tr)
cw = compute_class_weight(class_weight='balanced', classes=classes, y=y_tr)
class_weight_bin = {int(c): float(w) for c, w in zip(classes, cw)}

input_dim = X_tr.shape[1]

def build_binary_model(input_dim, n_units1, n_units2, dropout1, dropout2):
    model = Sequential([
        Dense(n_units1, activation='relu', input_dim=input_dim),
        Dropout(dropout1),
        Dense(n_units2, activation='relu'),
        Dropout(dropout2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss='binary_crossentropy'
    )
    return model

def objective_binary(trial):
    tf.keras.backend.clear_session()

    n_units1 = trial.suggest_int("n_units1", 32, 96, step=32)
    n_units2 = trial.suggest_int("n_units2", 16, 48, step=16)
    dropout1 = trial.suggest_float("dropout1", 0.1, 0.5)
    dropout2 = trial.suggest_float("dropout2", 0.1, 0.5)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    model = build_binary_model(input_dim, n_units1, n_units2, dropout1, dropout2)

    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=batch_size,
        callbacks=[es],
        class_weight=class_weight_bin,
        verbose=0
    )

    # Evaluate F2 at default 0.5 on validation
    y_proba = model.predict(X_val, verbose=0).ravel()
    y_hat = (y_proba >= 0.5).astype(int)
    f2 = fbeta_score(y_val, y_hat, beta=2, zero_division=0)
    return f2

sampler = optuna.samplers.TPESampler(seed=TUNE_SEED)
study_bin = optuna.create_study(direction="maximize", sampler=sampler)
study_bin.optimize(objective_binary, n_trials=10)

best_params_bin = study_bin.best_params
print("BINARY best F2:", study_bin.best_value)
print("BINARY best params:", best_params_bin)

Binary X: (1191438, 14)
Binary y counts:
 breast_cancer_history
0    969662
1    221776
Name: count, dtype: int64


BINARY best F2: 0.6481907239293894
BINARY best params: {'n_units1': 32, 'n_units2': 48, 'dropout1': 0.4100531293444458, 'dropout2': 0.4757995766256756, 'batch_size': 512}


In [ ]:
def tune_threshold_f2(y_true, y_proba, thresholds=np.linspace(0.1, 0.9, 17)):
    best = {"thr": 0.5, "f2": -1}
    for t in thresholds:
        y_hat = (y_proba >= t).astype(int)
        f2 = fbeta_score(y_true, y_hat, beta=2, zero_division=0)
        if f2 > best["f2"]:
            best = {"thr": float(t), "f2": float(f2)}
    return best

def run_binary_once(seed: int):
    set_global_seed(seed)
    tf.keras.backend.clear_session()

    X_train, X_test, y_train, y_test = train_test_split(
        X_bin, y_bin, test_size=0.2, stratify=y_bin, random_state=seed
    )
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=seed
    )

    # class weights from training only
    classes = np.unique(y_tr)
    cw = compute_class_weight(class_weight='balanced', classes=classes, y=y_tr)
    class_weight = {int(c): float(w) for c, w in zip(classes, cw)}

    model = build_binary_model(
        input_dim=X_tr.shape[1],
        n_units1=best_params_bin["n_units1"],
        n_units2=best_params_bin["n_units2"],
        dropout1=best_params_bin["dropout1"],
        dropout2=best_params_bin["dropout2"]
    )

    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=200,
        batch_size=best_params_bin["batch_size"],
        callbacks=[es],
        class_weight=class_weight,
        verbose=0
    )

    # threshold tuning on VAL
    y_proba_val = model.predict(X_val, verbose=0).ravel()
    best_thr_info = tune_threshold_f2(y_val, y_proba_val)
    thr = best_thr_info["thr"]

    # test eval
    y_proba_test = model.predict(X_test, verbose=0).ravel()
    y_hat_test = (y_proba_test >= thr).astype(int)

    acc  = accuracy_score(y_test, y_hat_test)
    rec  = recall_score(y_test, y_hat_test, zero_division=0)
    prec = precision_score(y_test, y_hat_test, zero_division=0)
    f1   = f1_score(y_test, y_hat_test, zero_division=0)

    cm = confusion_matrix(y_test, y_hat_test)
    tn, fp, fn, tp = cm.ravel()
    fnr = fn / (fn + tp + 1e-12)

    return {
        "seed": seed, "thr": thr,
        "accuracy": acc, "recall": rec, "precision": prec, "f1": f1, "fnr": fnr,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp
    }

SEEDS = [42, 123, 2025, 7, 999]   # use 3 if needed

bin_runs = []
for s in SEEDS:
    out = run_binary_once(s)
    bin_runs.append(out)
    print(f"[BINARY] seed={s} thr={out['thr']:.2f} recall={out['recall']:.4f}")

df_bin_runs = pd.DataFrame(bin_runs)
display(df_bin_runs)

summary_bin = df_bin_runs[["accuracy","precision","recall","f1","fnr"]].agg(["mean","std"]).T
summary_bin["mean±std"] = summary_bin["mean"].map(lambda x: f"{x:.4f}") + " ± " + summary_bin["std"].map(lambda x: f"{x:.4f}")
summary_bin

[BINARY] seed=42 thr=0.40 recall=0.8928


[BINARY] seed=123 thr=0.40 recall=0.8745


[BINARY] seed=2025 thr=0.40 recall=0.8919


[BINARY] seed=7 thr=0.45 recall=0.8733


[BINARY] seed=999 thr=0.40 recall=0.8797


## Three-class model, three seeds

In [3]:
import os, random
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

def set_global_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def build_mc_model(input_dim, n_classes, best_params_mc):
    model = Sequential([
        Dense(best_params_mc["n1"], activation='relu', input_dim=input_dim),
        Dropout(best_params_mc["d1"]),
        Dense(best_params_mc["n2"], activation='relu'),
        Dropout(best_params_mc["d2"]),
        Dense(n_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss='sparse_categorical_crossentropy'
    )
    return model

def run_multiclass_ann_once(X_mc, y_mc, seed: int, best_params_mc: dict):
    set_global_seed(seed)
    tf.keras.backend.clear_session()

    # Split: Train/Test
    X_train, X_test, y_train, y_test = train_test_split(
        X_mc, y_mc, test_size=0.2, stratify=y_mc, random_state=seed
    )

    # Split: Train/Val
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=seed
    )

    # Class weights from training only
    classes = np.unique(y_tr)
    cw = compute_class_weight(class_weight='balanced', classes=classes, y=y_tr)
    class_weight = {int(c): float(w) for c, w in zip(classes, cw)}

    n_classes = len(np.unique(y_mc))
    model = build_mc_model(input_dim=X_tr.shape[1], n_classes=n_classes, best_params_mc=best_params_mc)

    es = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

    model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=200,
        batch_size=best_params_mc["bs"],
        callbacks=[es],
        class_weight=class_weight,
        verbose=0
    )

    # Test evaluation
    y_test_pred = model.predict(X_test, verbose=0).argmax(axis=1)

    acc = accuracy_score(y_test, y_test_pred)
    macro_rec = recall_score(y_test, y_test_pred, average="macro", zero_division=0)
    macro_prec = precision_score(y_test, y_test_pred, average="macro", zero_division=0)
    macro_f1 = f1_score(y_test, y_test_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_test, y_test_pred)

    return {
        "seed": seed,
        "accuracy": float(acc),
        "macro_precision": float(macro_prec),
        "macro_recall": float(macro_rec),
        "macro_f1": float(macro_f1),
        "cm": cm
    }

In [4]:
from sklearn.linear_model import LogisticRegression

# --- Build multiclass dataset ---

df_mc = df.copy()   # df = preprocessed dataframe you already have

# Binary target
y_bin_full = df_mc['breast_cancer_history'].astype('int32')
X_logit = df_mc.drop(columns=['breast_cancer_history']).astype('float32')

# Logistic regression to estimate risk probability
set_global_seed(42)
logit = LogisticRegression(max_iter=5000)
logit.fit(X_logit, y_bin_full)

df_mc['pred_prob'] = logit.predict_proba(X_logit)[:, 1]

# Create 3-class target
df_mc['breast_cancer_history_3class'] = y_bin_full.copy()

LOWER = 0.35
UPPER = 0.50

mask_border = (
    (df_mc['breast_cancer_history_3class'] == 0) &
    (df_mc['pred_prob'] >= LOWER) &
    (df_mc['pred_prob'] <= UPPER)
)

df_mc.loc[mask_border, 'breast_cancer_history_3class'] = 2

# Drop unused columns
df_mc = df_mc.drop(columns=['breast_cancer_history', 'pred_prob'])

# Final multiclass X and y
y_mc = df_mc['breast_cancer_history_3class'].astype('int32')
X_mc = df_mc.drop(columns=['breast_cancer_history_3class']).astype('float32')

print("X_mc shape:", X_mc.shape)
print("y_mc distribution:")
print(y_mc.value_counts())

X_mc shape: (1191438, 14)
y_mc distribution:
breast_cancer_history_3class
0    888887
1    221776
2     80775
Name: count, dtype: int64


In [5]:
# Multiclass hyperparameter tuning using Optuna
input_dim_mc = X_mc.shape[1]
n_classes_mc = len(np.unique(y_mc))

def objective_multiclass(trial):
    tf.keras.backend.clear_session()

    # Hyperparameters to tune
    n1 = trial.suggest_int("n1", 32, 96, step=32)
    n2 = trial.suggest_int("n2", 16, 48, step=16)
    d1 = trial.suggest_float("d1", 0.1, 0.5)
    d2 = trial.suggest_float("d2", 0.1, 0.5)
    bs = trial.suggest_categorical("bs", [128, 256, 512])

    # Build model using trial parameters
    current_params = {"n1": n1, "n2": n2, "d1": d1, "d2": d2, "bs": bs}
    model = build_mc_model(input_dim_mc, n_classes_mc, current_params)

    # Re-split for the objective, using a fixed seed, to ensure reproducibility of tuning.
    TUNE_SEED = 42 # Reusing the TUNE_SEED from binary for consistency
    X_train_mc, _, y_train_mc, _ = train_test_split(
        X_mc, y_mc, test_size=0.2, stratify=y_mc, random_state=TUNE_SEED
    )
    X_tr_mc, X_val_mc, y_tr_mc, y_val_mc = train_test_split(
        X_train_mc, y_train_mc, test_size=0.2, stratify=y_train_mc, random_state=TUNE_SEED
    )

    # Class weights for tuning
    classes_mc = np.unique(y_tr_mc)
    cw_mc = compute_class_weight(class_weight='balanced', classes=classes_mc, y=y_tr_mc)
    class_weight_mc = {int(c): float(w) for c, w in zip(classes_mc, cw_mc)}

    es = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

    model.fit(
        X_tr_mc, y_tr_mc,
        validation_data=(X_val_mc, y_val_mc),
        epochs=50, # Shorter epochs for tuning
        batch_size=bs, # Use trial batch_size
        callbacks=[es],
        class_weight=class_weight_mc,
        verbose=0
    )

    # Evaluate using macro F1 score on validation set for multiclass
    y_proba_val = model.predict(X_val_mc, verbose=0).argmax(axis=1)
    macro_f1 = f1_score(y_val_mc, y_proba_val, average="macro", zero_division=0)
    return macro_f1

sampler_mc = optuna.samplers.TPESampler(seed=42)
study_mc = optuna.create_study(direction="maximize", sampler=sampler_mc)
study_mc.optimize(objective_multiclass, n_trials=10)

best_params_mc = study_mc.best_params
print("MULTICLASS best F1:", study_mc.best_value)
print("MULTICLASS best params:", best_params_mc)

SEEDS_MC = [42, 123, 2025]

runs = []
cms = []

for s in SEEDS_MC:
    out = run_multiclass_ann_once(X_mc, y_mc, seed=s, best_params_mc=best_params_mc)
    runs.append({k:v for k,v in out.items() if k != "cm"})
    cms.append(out["cm"])
    print(f"[MULTICLASS] seed={s} macro_recall={out['macro_recall']:.4f}")

df_runs = pd.DataFrame(runs)
display(df_runs)

summary_mc = df_runs[["accuracy","macro_precision","macro_recall","macro_f1"]].agg(["mean","std"]).T
summary_mc["mean±std"] = summary_mc["mean"].map(lambda x: f"{x:.4f}") + " ± " + summary_mc["std"].map(lambda x: f"{x:.4f}")
summary_mc

MULTICLASS best F1: 0.6653703150074989
MULTICLASS best params: {'n1': 64, 'n2': 16, 'd1': 0.31707843326329943, 'd2': 0.15636968998990508, 'bs': 512}


[MULTICLASS] seed=42 macro_recall=0.7552


[MULTICLASS] seed=123 macro_recall=0.7529


[MULTICLASS] seed=2025 macro_recall=0.7555


In [6]:
y_mc.value_counts(normalize=True)

,proportion
breast_cancer_history_3class,
0,0.746062
1,0.186141
2,0.067796
